# 01 — Rule-based NLP: tokenization, regex, lexicons, precision and recall

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Rule-based NLP uses human-written logic. It is explainable and easy to ship for narrow tasks, but it does not scale well to messy language.

Good uses:

- email, URL, date, money extraction
- compliance checks
- deterministic guardrails
- MVP bootstrapping before you have labels

Bad uses:

- sarcasm-heavy sentiment
- broad semantic understanding
- open-ended reasoning

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import re
import pandas as pd
from collections import Counter

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
df[["text", "sentiment", "intent"]].head()

## Tokenization from scratch

This simple tokenizer is not production-grade, but it makes tokenization transparent. Modern production systems usually use library tokenizers or subword tokenizers.

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:\.\d+)?|[^\w\s]")

def simple_tokenize(text):
    return TOKEN_RE.findall(text.lower())

for text in df["text"].head(3):
    print(text)
    print(simple_tokenize(text))
    print()

## Regex extraction

Rules work well when the pattern is explicit. For company products, these rules are often high-value glue code around LLMs.

In [ ]:
examples = [
    "Email jp@example.com before 5pm on 2026-07-01.",
    "The invoice total is $1,249.99 and is due on 16/06/2026.",
    "Contact support+vip@company.ai for SSO issues.",
]

patterns = {
    "email": re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"),
    "money": re.compile(r"\$\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?"),
    "iso_date": re.compile(r"\b\d{4}-\d{2}-\d{2}\b"),
    "slash_date": re.compile(r"\b\d{1,2}/\d{1,2}/\d{4}\b"),
}

for text in examples:
    print("TEXT:", text)
    for name, pat in patterns.items():
        print(" ", name, pat.findall(text))
    print()

## Lexicon sentiment baseline

A lexicon counts positive and negative words. We will add three small upgrades:

1. negation handling: "not good" should be negative
2. intensifiers: "very good" is stronger
3. contrast handling: after "but", later sentiment often dominates

In [ ]:
positive_words = {"love", "great", "excellent", "faster", "solved", "like", "clear", "fair", "saves", "needed"}
negative_words = {"crashes", "crash", "charged", "confusing", "stuck", "wrong", "outage", "slow", "missed", "invented", "cannot", "cancel"}
negations = {"not", "never", "no", "cannot", "can't", "wont", "won't"}
intensifiers = {"very", "really", "extremely", "completely"}

def lexicon_sentiment(text):
    toks = simple_tokenize(text)
    # If there is a contrast marker, weight text after it more strongly.
    if "but" in toks:
        idx = toks.index("but")
        toks = toks[:idx] + toks[idx+1:] * 2
    score = 0
    for i, tok in enumerate(toks):
        weight = 1
        if i > 0 and toks[i-1] in intensifiers:
            weight = 2
        sign = 0
        if tok in positive_words:
            sign = 1
        elif tok in negative_words:
            sign = -1
        if sign != 0 and i > 0 and toks[i-1] in negations:
            sign *= -1
        score += weight * sign
    if score > 0:
        return "positive"
    if score < 0:
        return "negative"
    return "neutral"

texts = [
    "I love the product",
    "I do not love the product",
    "The mobile app is slow but the web app works well",
    "Great, another outage during our board meeting",
]
for t in texts:
    print(t, "=>", lexicon_sentiment(t))

## Evaluate the rule baseline

Use precision and recall to understand where rules are safe. For example, you may use high-precision rules to escalate obvious complaints, but not rely on them for all sentiment.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = df["sentiment"]
y_pred = df["text"].apply(lexicon_sentiment)
print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred, labels=["negative", "neutral", "positive"]))

errors = df.assign(pred=y_pred).loc[lambda d: d["sentiment"] != d["pred"], ["text", "sentiment", "pred"]]
errors

## Optional: spaCy pipeline

Use spaCy when you need industrial tokenization, POS tagging, dependency parsing, NER, and pipeline components. The blank model below works without downloading a trained model; replace it with `en_core_web_sm` or a domain model for production.

In [ ]:
try:
    import spacy
    nlp = spacy.blank("en")
    doc = nlp("Commonwealth Bank opened an office in Sydney.")
    print([token.text for token in doc])
except Exception as e:
    print("spaCy not installed or not available:", e)

## Company-data adaptation

Replace the lexicons with domain-specific terms. For example:

- fintech: "chargeback", "fraud", "settlement", "KYC"
- legal: "indemnity", "termination", "liability", "jurisdiction"
- healthcare: "adverse", "allergy", "contraindication", "urgent"

Rules are most useful when paired with error analysis and human review.